# **A05 - Pipeline**

---
Nombre: **Alan Jesús Hernández Soto** \
Carrera: **Ingeniería Financiera** \
Materia: **Laboratorio de Aprendizaje Estadístico** \
Tipo de Trabajo: **Actividad**

---

## Índice

1. Carga y análisis exploratorio de datos
2. Preparación del pipeline
3. Entrenamiento y evaluación inicial
4. Análisis de residuos
5. Validación cruzada

## 1. Carga y análisis exploratorio de datos

Se carga el conjunto *Motor Trend Car Road Tests*. Cada observación representa un automóvil y se busca estimar su rendimiento de combustible, medido en millas por galón (`mpg`), a partir de sus características mecánicas.

**Importación de librerías**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

**Carga y revisión inicial**

Se verifica el tamaño, los tipos de datos y las primeras observaciones antes de preparar las variables para el modelo.

In [ ]:
df = pd.read_excel('Datasets/Motor Trend Car Road Tests.xlsx', sheet_name='mtcars')

df.info()
display(df.head())

**Diccionario y tipo de variables**

La siguiente tabla resume el significado de cada variable y el tratamiento que recibirá. Aunque algunas categorías están codificadas con números, no se interpretan como magnitudes continuas.

In [ ]:
variable_info = pd.DataFrame({
    'Variable': ['model', 'mpg', 'cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am', 'gear', 'carb'],
    'Descripción': [
        'Nombre o identificador del automóvil',
        'Rendimiento de combustible en millas por galón estadounidense',
        'Número de cilindros del motor',
        'Cilindrada del motor en pulgadas cúbicas',
        'Potencia bruta del motor en caballos de fuerza',
        'Relación del eje trasero',
        'Peso del automóvil en miles de libras',
        'Tiempo para recorrer un cuarto de milla, en segundos',
        'Configuración del motor: 0 = V, 1 = en línea',
        'Transmisión: 0 = automática, 1 = manual',
        'Número de marchas hacia adelante',
        'Número de carburadores'
    ],
    'Tipo': [
        'Categórica nominal', 'Numérica continua (objetivo)', 'Categórica ordinal',
        'Numérica continua', 'Numérica discreta', 'Numérica continua',
        'Numérica continua', 'Numérica continua', 'Booleana', 'Booleana',
        'Categórica ordinal', 'Categórica ordinal'
    ],
    'Tratamiento en el pipeline': [
        'Se elimina: solo identifica la observación', 'Variable respuesta', 'Codificación one-hot',
        'Estandarización', 'Estandarización', 'Estandarización',
        'Estandarización', 'Estandarización', 'Se conserva como 0/1', 'Se conserva como 0/1',
        'Codificación one-hot', 'Codificación one-hot'
    ]
})

display(variable_info)

**Preparación de predictores y respuesta**

`model` se excluye porque es un identificador. Se separa `mpg` como respuesta y se agrupan los predictores según su tipo para transformarlos correctamente.

In [ ]:
target_col = 'mpg'
categorical_cols = ['cyl', 'gear', 'carb']
boolean_cols = ['vs', 'am']
numerical_cols = ['disp', 'hp', 'drat', 'wt', 'qsec']

X = df.drop(columns=['model', target_col])
y = df[target_col]

## 2. Preparación del pipeline

El *pipeline* concentra las transformaciones y el modelo en un solo objeto. Las variables numéricas se estandarizan, las categóricas se codifican y las booleanas se mantienen como indicadores 0/1.

**Transformación y modelo**

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('bool', 'passthrough', boolean_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

## 3. Entrenamiento y evaluación inicial

Se ajusta una regresión lineal para predecir `mpg`. El coeficiente de determinación $R^2$ en entrenamiento indica qué proporción de la variación del rendimiento explica el modelo en estos mismos datos.

**Ajuste del modelo**

In [ ]:
pipeline.fit(X, y)

r2_entrenamiento = pipeline.score(X, y)
print(f'R² en entrenamiento: {r2_entrenamiento:.3f}')

## 4. Análisis de residuos

Los residuos son la diferencia entre el rendimiento observado y el predicho. Una dispersión aleatoria alrededor de cero es consistente con un ajuste lineal adecuado; patrones o cambios marcados en la dispersión sugieren que el modelo puede mejorarse.

In [ ]:
y_pred = pipeline.predict(X)
residuos = y - y_pred

plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuos)
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residuos vs. valores predichos')
plt.xlabel('Valores predichos de mpg')
plt.ylabel('Residuos')
plt.show()

## 5. Validación cruzada

La validación cruzada estima el desempeño fuera de muestra: en cada partición se entrena el pipeline con cuatro quintas partes de los autos y se evalúa con la quinta parte restante. Se reporta un valor de $R^2$ por partición y su promedio.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores_cv = cross_val_score(
    estimator=pipeline,
    X=X,
    y=y,
    cv=cv,
    scoring='r2'
)

print('R² por partición:', scores_cv)
print(f'R² promedio: {scores_cv.mean():.3f}')